# Calorie Calculator and HR Zones Importer

This notebook calculates calorie burn rates based on heart rate data and imports them into the database. It also ensures HR zones are properly configured.

## Process
1.  **Load Configuration**: Loads database and path settings from environment variables.
2.  **Import HR Zones**: Ensures the `hr_zones` table exists and is populated from `zones.csv`.
3.  **Load VO2max Data**: Reads VO2max/Calorie data from a CSV file (`v02max_data.csv`).
4.  **Expand & Interpolate**: Expands the data to fill in missing heart rate values using linear interpolation.
5.  **Filter & Collapse**: 
    -   Slices the data up to the maximum heart rate (ignoring cool-down phase).
    -   Sorts by HR.
    -   Collapses consecutive duplicate HR values by averaging calories.
6.  **Import Calories**: Imports the processed data into the `calories_per_hr` table.

## Usage
- `ensure_hr_zones_table(config)`: Imports HR zones from zones.csv
- `calculate_and_import_calories(path, config)`: Calculates and imports calorie data

In [ ]:
import sys
from pathlib import Path

# Add src to path
repo_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(repo_root))

import importlib
from polar.storage import duckdb
from polar.storage import postgres
from polar.utils.config import load_configuration

# Reload the modules to ensure we have the latest changes
importlib.reload(duckdb)
importlib.reload(postgres)

# Load configuration
config = load_configuration()

# Define path to VO2max data
# The data file is in data/v02max_data.csv relative to repo root
v02max_data_path = Path('../data/v02max_data.csv')

# Get database type from configuration
db_type = config.get('DATABASE_TYPE', 'duckdb')

print(f"📊 Using {db_type.upper()} database")
print("="*60)

if db_type == 'postgres':
    from polar.storage.postgres import ensure_hr_zones_table, calculate_and_import_calories
    print("\n📈 Step 1: Importing HR zones to Postgres...")
    ensure_hr_zones_table(config)
    print("\n🔥 Step 2: Calculating and importing calorie data to Postgres...")
    calculate_and_import_calories(v02max_data_path, config)
else:  # default to duckdb
    from polar.storage.duckdb import ensure_hr_zones_table, calculate_and_import_calories
    print("\n📈 Step 1: Importing HR zones to DuckDB...")
    ensure_hr_zones_table(config)
    print("\n🔥 Step 2: Calculating and importing calorie data to DuckDB´...")
    calculate_and_import_calories(v02max_data_path, config)

print("\n" + "="*60)
print("✅ All imports completed successfully!")

/Users/tonkata/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Configuration loaded
  - Client ID: fddbcde4...
  - Redirect Port: 5001
  - Member ID: 61732059
  - Database Type: POSTGRES
  - DuckDB Path: /Users/tonkata/repos/workoutdata/hr_data/database_v2.duckdb
  - Tokens File: /Users/tonkata/repos/workoutdata/notebooks/tokens_polar.json
  - VO2max Data: /Users/tonkata/repos/workoutdata/data/v02max_data.csv
  - Zones CSV: /Users/tonkata/repos/workoutdata/hr_data/zones.csv
  - Output Dir: /Users/tonkata/repos/workoutdata/hr_data
  - PostgreSQL: humandcoded-pg.postgres.database.azure.com/workoutdata
  - Azure Storage: muskulsa/workoutdata
📊 Using POSTGRES database

📈 Step 1: Importing HR zones...
✅ HR zones table created with 5 zones

🔥 Step 2: Calculating and importing calorie data...
Reading VO2max data from ../data/v02max_data.csv...
Total rows in original DataFrame: 85
Total rows in expanded DataFrame: 126
Maximum HR value in expanded_df: 185.0 at index 72
Total rows in expanded sliced (0:72) DataFrame : 114

Expanded data from HR rise (inde

,HR,Calories,Calories_Second
0,96.000000,2.330000,0.038833
1,97.000000,2.485000,0.041417
2,98.000000,2.640000,0.044000
3,99.000000,2.795000,0.046583
4,100.000000,2.950000,0.049167
5,101.000000,3.105000,0.051750
6,102.000000,3.260000,0.054333
7,103.000000,3.415000,0.056917
8,104.000000,3.570000,0.059500
9,105.000000,3.725000,0.062083


Total rows in collapsed DataFrame: 90

Full collapsed DataFrame:


,HR,Calories,Calories_Second
0,96.000000,2.330000,0.038833
1,97.000000,2.485000,0.041417
2,98.000000,2.640000,0.044000
3,99.000000,2.795000,0.046583
4,100.000000,2.950000,0.049167
5,101.000000,3.105000,0.051750
6,102.000000,3.260000,0.054333
7,103.000000,3.415000,0.056917
8,104.000000,3.570000,0.059500
9,105.000000,3.725000,0.062083


ℹ️  Database 'workoutdata' already exists
✅ Data imported to PostgreSQL table 'calories_per_hr' with 90 rows

✅ All imports completed successfully!
